# Detección de ejercicios y técnica — Práctica 3

Reconoce **sentadilla, lagartija y curl de bíceps**, cuenta las repeticiones y avisa si cada una está **bien** o **mal** hecha.

**Cómo funciona**
1. MediaPipe detecta 33 puntos del cuerpo en cada cuadro.
2. Con esos puntos se calculan 8 señales: 6 ángulos (rodilla, cadera, codo, hombro, inclinación del torso y línea hombro-cadera-tobillo) y 2 medidas normalizadas (profundidad de la cadera y rodilla delante del pie). El lado del cuerpo se fija con histéresis y las señales se suavizan con una mediana móvil.
3. **Modelo de ejercicio:** clasifica ventanas de 2 s de señales (media, desviación, percentiles 5 y 95).
4. Las repeticiones se cortan con el ángulo principal (rodilla en sentadilla, codo en lagartija y curl).
5. **Modelo de técnica** (uno por ejercicio): clasifica cada repetición como `bien` o `mal` (mínimo, máximo, media, desviación y rango de cada ángulo, más la duración), y da un veredicto por serie promediando sus repeticiones.

**Evaluación:** se aparta ~20 % de las personas como prueba final. Con el resto se comparan 8 candidatos (Dummy, una regla de un umbral, regresión logística, Random Forest, HistGradientBoosting, SVM, MLP y ROCKET) con validación cruzada anidada y agrupada por persona; el modelo calibra sus probabilidades. En vivo, además del veredicto de cada repetición, se da uno por serie (más confiable). Luego se mide el sistema completo con la prueba final y se corren ablaciones y curvas de aprendizaje.

**Datos:** se entrena con el dataset público *Multi-View Fitness Exercises with Good/Bad Form Labels* (Mendeley Data, 26 personas, CC BY 4.0), vista de perfil. Se pueden agregar videos propios en la misma carpeta.

**Estructura en Google Drive** (la crea el notebook)
```
MyDrive/deteccion_ejercicios/
  videos/
    mendeley_subject_001/sentadilla__bien__side.mp4   <- del dataset (paso 3)
    horta_andrea/sentadilla__poco_profunda__1.mp4     <- videos propios (PROTOCOLO_GRABACION.md)
  pruebas/        <- videos para probar el modelo ya entrenado
```
Nombre de los videos propios: `ejercicio__etiqueta__n.mp4`, con ejercicio = `sentadilla`, `lagartija` o `curl` y etiqueta = `bien`, `mal` o un error específico (`poco_profunda`, `torso_inclinado`, `cadera_caida`, `balanceo`, `recorrido_parcial`). Cómo grabarlos: `PROTOCOLO_GRABACION.md`.

Si la carpeta te la compartieron, en Drive haz clic derecho → *Organizar → Agregar acceso directo* a "Mi unidad" para que Colab la vea.

## 1. Preparar el entorno

In [ ]:
REPO = "https://github.com/AndreaHortaHernandez/Deteccion-de-ejercicios.git"
import os
if os.path.exists("/content/Deteccion-de-ejercicios"):
    !git -C /content/Deteccion-de-ejercicios pull -q
else:
    !git clone -q $REPO /content/Deteccion-de-ejercicios
%cd /content/Deteccion-de-ejercicios
# --no-deps: evita que MediaPipe instale una segunda copia de OpenCV encima de la de Colab
!pip install -q --no-deps mediapipe==1.2.2
!pip install -q flatbuffers
import pose
pose.asegurar_modelo_pose()
print("Listo")

## 2. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/deteccion_ejercicios"   # cámbialo si su carpeta tiene otro nombre
VIDEOS = f"{BASE}/videos"
DATOS = f"{BASE}/datos"        # aquí se guardan los puntos de pose (CSV); así no se recalculan
MODELOS = f"{BASE}/modelos"
SALIDAS = f"{BASE}/salidas"
for p in (DATOS, MODELOS, SALIDAS):
    os.makedirs(p, exist_ok=True)
!ls "$VIDEOS"

## 3. Descargar el dataset de Mendeley
Baja la vista de perfil de los 3 ejercicios (~148 videos, ~512 MB) directo a tu Drive. Solo descarga lo que falta, así que si se corta vuelve a correr la celda.

Cita: Duddela Sai Prashanth et al. (2026). *A Multi-View Raw Video Dataset of Seven Fitness Exercises with Good/Bad Form Labels*. Mendeley Data, v2. https://doi.org/10.17632/kgbb3yn47p.2

In [ ]:
!python importar_mendeley.py --videos "$VIDEOS"

### 3b. Videos propios
Sigan [PROTOCOLO_GRABACION.md](PROTOCOLO_GRABACION.md): 9 videos por persona, de perfil, horizontal.

**Recomendado: grabar con la webcam de una laptop.** En la laptop (no en Colab), doble clic en `grabar.bat` (o `python grabar_webcam.py`). La pantalla guía cada paso y graba sola; los videos salen ya con su nombre en `videos/<apellido_nombre>/`, junto con `conteos.csv` (las reps contadas). Cada quien sube su carpeta `apellido_nombre` completa a la carpeta compartida del equipo **practica 3 gym**. Quien corra el notebook le da *Agregar acceso directo a Mi unidad* a esa carpeta y corre la celda de abajo, que copia los videos a `VIDEOS/`; después, solo la celda de **revisar** (la última de esta sección). Las reps contadas se pasan solas a `DATOS/conteos_reales.csv` en el paso 4.

**Si graban con el celular:**
1. **Guía:** abran este notebook en una laptop junto al celular y corran la primera celda con el nombre y número de la persona. Dice cuándo grabar, quedarse quieto, hacer las reps y detener, y guarda las reps en `DATOS/conteos_reales.csv`.
2. **Renombrar:** suban los videos del celular, tal cual, a `BASE/celular/<apellido_nombre>/` (solo los 9 buenos) y corran la segunda celda; con `--aplicar` los mueve a `VIDEOS/<apellido_nombre>/`.
3. **Revisar:** la tercera celda avisa si falta algún video o si alguno está vertical, en cámara lenta o demasiado corto.

In [ ]:
# Copia las carpetas de cada persona desde la carpeta compartida del equipo (no sobrescribe lo que ya está)
EQUIPO = "/content/drive/MyDrive/practica 3 gym"   # acceso directo en Mi unidad a la carpeta compartida
!cp -rn "$EQUIPO"/*/ "$VIDEOS/" && ls "$VIDEOS"

In [ ]:
PERSONA, NUM = "perez_ana", 1   # apellido_nombre (minúsculas, sin acentos) y su número del 1 al 5
!python guia_grabacion.py grabar --persona $PERSONA --num $NUM --conteos "$DATOS/conteos_reales.csv"

In [ ]:
# misma PERSONA y NUM de la celda de arriba (cámbienlos para cada persona)
!python guia_grabacion.py renombrar --origen "$BASE/celular/$PERSONA" --persona $PERSONA --num $NUM --videos "$VIDEOS"
# si la tabla está bien, corran otra vez agregando --aplicar al final de la línea de arriba

In [ ]:
!python guia_grabacion.py revisar --videos "$VIDEOS"

## 4. Extraer la pose de los videos
Solo procesa videos nuevos o modificados (la primera vez tarda unos minutos). Al final lista los videos con problemas (nombre mal escrito, persona fuera de cuadro…).
También crea `DATOS/conteos_reales.csv` con una fila por video: anoten ahí las repeticiones reales (columna `reps_reales`).

In [ ]:
!python extraer_videos.py --videos "$VIDEOS" --datos "$DATOS"

## 5. Entrenar y evaluar
Compara los candidatos con validación cruzada anidada (agrupada por persona), mide al ganador en las personas de prueba final y guarda:
- `MODELOS/`: modelos entrenados con todas las personas (para usar en vivo).
- `MODELOS/evaluacion/`: modelos entrenados **sin** las personas de prueba (para el paso 6).
- `SALIDAS/`: `reporte_entrenamiento.txt`, `resultados.csv` y gráficos (matrices normalizadas, curvas PR/ROC, confiabilidad, acierto por persona e importancia por permutación).

La corrida completa tarda unos 20-30 min en Colab. Para probar rápido agrega `--rapido` (sin búsqueda de hiperparámetros, 1 repetición).

In [ ]:
!python entrenar.py --datos "$DATOS" --modelos "$MODELOS" --salidas "$SALIDAS"

In [ ]:
import glob, os
from IPython.display import Image, display

def mostrar(*patrones):
    for patron in patrones:
        for png in sorted(glob.glob(f"{SALIDAS}/{patron}")):
            print(os.path.basename(png))
            display(Image(png))

mostrar("confusion_*_cv.png", "confusion_*_prueba.png", "pr_roc_*.png", "confiabilidad_*.png",
        "personas_*.png", "importancia_*.png")

## 6. Evaluar el sistema completo
Pasa cada video de las personas de prueba por el analizador real, como en vivo: ejercicio reconocido, conteo de repeticiones y técnica por video.

Para medir el **error de conteo**, abre `SALIDAS/conteos_para_llenar.csv`, anota las repeticiones reales en la columna `reps_reales` (basta con una parte de los videos) y guárdalo como `DATOS/conteos_reales.csv`. Después vuelve a correr esta celda.

In [ ]:
!python evaluar_sistema.py --datos "$DATOS" --modelos "$MODELOS" --salidas "$SALIDAS"
mostrar("sistema_confusion_*.png", "linea_de_tiempo.png", "sensibilidad_conteo.png")

## 7. Experimentos para el reporte
Solo usan las personas de entrenamiento:
- **Comparación v1 vs. v2:** mismos modelos y mismas particiones por persona, con Wilcoxon pareado.
- **Ablaciones:** cuánto aporta cada decisión (suavizado, lado fijo, percentiles, largo de ventana, familias de rasgos, ROCKET) frente a la versión 1.
- **Curva de aprendizaje** por número de personas: ¿faltan datos o falta modelo?
- **Robustez al punto de vista:** modelos entrenados de perfil, probados con la vista diagonal. Para esto primero corre la celda opcional de abajo.

In [ ]:
# Opcional: vista diagonal del dataset para la prueba de robustez (~500 MB más)
# !python importar_mendeley.py --videos "$VIDEOS" --vistas side,diagonal
# !python extraer_videos.py --videos "$VIDEOS" --datos "$DATOS"

In [ ]:
!python experimentos.py --datos "$DATOS" --modelos "$MODELOS" --salidas "$SALIDAS"
mostrar("ablaciones_*.png", "curva_aprendizaje_*.png")
print(open(f"{SALIDAS}/reporte_experimentos.txt").read())

## 8. Probar con un video
Usa un video que **no** se haya usado para entrenar (ponlo en `pruebas/` o súbelo aquí).

In [ ]:
VIDEO_PRUEBA = f"{BASE}/pruebas/prueba1.mp4"   # o deja "" para subir uno desde tu compu

if not VIDEO_PRUEBA:
    from google.colab import files
    subido = files.upload()
    VIDEO_PRUEBA = "/content/" + next(iter(subido))

!python analizar_video.py --video "$VIDEO_PRUEBA" --salida /content/analizado.mp4 --modelos "$MODELOS"
# convertir a H.264 para que el navegador lo pueda reproducir
!ffmpeg -y -loglevel error -i /content/analizado.mp4 -vcodec libx264 -pix_fmt yuv420p /content/analizado_h264.mp4
from IPython.display import Video
Video("/content/analizado_h264.mp4", embed=True, width=640)

## 9. Cámara en vivo
Colab pide permiso para usar la cámara. Colócate **de perfil** y con el cuerpo completo en cuadro.
Va a unos cuantos cuadros por segundo (cada cuadro viaja al servidor de Colab), así que haz las repeticiones a ritmo normal o un poco lento.
Haz clic sobre el video para detener.

In [ ]:
import importlib, colab_camara
importlib.reload(colab_camara)
analizador = colab_camara.correr(MODELOS)